# Virtual Cell MVP

This notebook builds a minimal bidirectional retrieval prototype over the bioRxiv marker extraction corpus.

A **profile** is one `(paper, cell type)` object with three linked parts:

- a cell-type label
- a set of marker genes
- the manuscript sentences that justify the marker claims

The notebook supports two query modes:

1. **Text query** over profile text
2. **Gene query** over the profile's marker-gene set, using either gene symbols or Ensembl IDs

This notebook now compares two text retrievers:

- a sparse TF-IDF baseline
- a cached biomedical sentence-transformer model (`pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb`)

This is a runnable prototype. It is not a trained multimodal model.

In [1]:
from __future__ import annotations

import json
import os
import re
import sqlite3
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def find_repo_root() -> Path:
    here = Path.cwd()
    candidates = [here, here.parent]
    for root in candidates:
        if (root / 'docs' / 'llmarkers.sqlite').exists():
            return root
    raise FileNotFoundError('Could not locate docs/llmarkers.sqlite from the current working directory.')


REPO_ROOT = find_repo_root()
DB_PATH = REPO_ROOT / 'docs' / 'llmarkers.sqlite'
print(f'Repo root: {REPO_ROOT}')
print(f'Database:  {DB_PATH}')

Repo root: /Users/sinabooeshaghi/projects/markergeneextraction/llmarkers
Database:  /Users/sinabooeshaghi/projects/markergeneextraction/llmarkers/docs/llmarkers.sqlite


In [2]:
query = '''
SELECT
    p.paper_id,
    p.doi,
    p.title,
    p.year,
    p.license,
    m.group_name,
    m.feature_name,
    m.feature_id,
    m.source_rationale,
    m.source_type
FROM papers AS p
JOIN markers AS m ON m.paper_id = p.paper_id
WHERE COALESCE(m.data_id, '') = ''
  AND COALESCE(m.group_name, '') <> ''
  AND COALESCE(m.feature_name, '') <> ''
ORDER BY p.paper_id, m.group_name, m.feature_name
'''

with sqlite3.connect(DB_PATH) as conn:
    rows = pd.read_sql_query(query, conn)

print(f'Loaded {len(rows):,} bioRxiv marker rows from SQLite.')
display(rows.head())

Loaded 12,516 bioRxiv marker rows from SQLite.


,paper_id,doi,title,year,license,group_name,feature_name,feature_id,source_rationale,source_type
0,4,10.1038/s41467-021-25968-8,Multi-species single-cell transcriptomic analy...,2021,NaN,RETINAL PIGMENTED EPITHELIUM,BEST1,ENSG00000167995,NaN,image
1,4,10.1038/s41467-021-25968-8,Multi-species single-cell transcriptomic analy...,2021,NaN,RETINAL PIGMENTED EPITHELIUM,BEST1,ENSG00000167995,NaN,image
2,4,10.1038/s41467-021-25968-8,Multi-species single-cell transcriptomic analy...,2021,NaN,RETINAL PIGMENTED EPITHELIUM,CNGB3,ENSG00000170289,NaN,image
3,4,10.1038/s41467-021-25968-8,Multi-species single-cell transcriptomic analy...,2021,NaN,RETINAL PIGMENTED EPITHELIUM,COL4A3,ENSG00000169031,NaN,image
4,4,10.1038/s41467-021-25968-8,Multi-species single-cell transcriptomic analy...,2021,NaN,RETINAL PIGMENTED EPITHELIUM,CRX,ENSG00000105392,NaN,image


In [3]:
def clean_text(value: str | None) -> str:
    if value is None:
        return ''
    return re.sub(r'\s+', ' ', str(value)).strip()


def dedupe_keep_order(values):
    out = []
    seen = set()
    for value in values:
        if not value:
            continue
        if value in seen:
            continue
        seen.add(value)
        out.append(value)
    return out


def normalize_doi(value: str | None) -> str:
    if not value:
        return ''
    text = clean_text(value).lower()
    for prefix in ('https://doi.org/', 'http://doi.org/', 'doi:'):
        if text.startswith(prefix):
            text = text[len(prefix):]
            break
    return text.rstrip('. )]')


def extract_abstracts_from_meca(root: Path) -> dict[str, str]:
    context = {}
    meca_root = root / 'data' / 'biorxiv' / 'meca'
    for folder in meca_root.iterdir():
        if not folder.is_dir():
            continue
        xml_paths = [
            p for p in folder.glob('*.xml')
            if p.name not in {'directives.xml', 'manifest.xml', 'transfer.xml'}
        ]
        if not xml_paths:
            continue
        xml_path = xml_paths[0]
        try:
            root_xml = ET.parse(xml_path).getroot()
        except ET.ParseError:
            continue

        doi = ''
        for node in root_xml.findall('.//article-id'):
            if node.attrib.get('pub-id-type') == 'doi' and node.text:
                doi = normalize_doi(node.text)
                if doi:
                    break
        if not doi:
            continue

        abstract_node = root_xml.find('.//abstract')
        if abstract_node is None:
            continue
        abstract_text = clean_text(' '.join(abstract_node.itertext()))
        if abstract_text:
            context[doi] = abstract_text
    return context


paper_context_by_doi = extract_abstracts_from_meca(REPO_ROOT)
print(f'Loaded title/abstract context for {len(paper_context_by_doi):,} papers from MECA XML.')


records = []
group_cols = ['paper_id', 'doi', 'title', 'year', 'license', 'group_name']
for key, grp in rows.groupby(group_cols, dropna=False, sort=False):
    paper_id, doi, title, year, license_text, group_name = key
    gene_names = dedupe_keep_order(clean_text(v).upper() for v in grp['feature_name'].tolist())
    gene_ids = dedupe_keep_order(clean_text(v).upper() for v in grp['feature_id'].fillna('').tolist())
    sentences = dedupe_keep_order(clean_text(v) for v in grp['source_rationale'].fillna('').tolist())
    label = clean_text(group_name)
    doi_norm = normalize_doi(doi)
    paper_context_blob = ' '.join(part for part in [clean_text(title), paper_context_by_doi.get(doi_norm, '')] if part)
    # Repeat the label to keep exact cell-type wording important during retrieval.
    text_blob = ' '.join([label, label, label] + sentences)
    records.append(
        {
            'profile_id': f'{paper_id}::{clean_text(group_name)}',
            'paper_id': int(paper_id),
            'doi': clean_text(doi),
            'title': clean_text(title),
            'year': int(year) if pd.notna(year) else None,
            'license': clean_text(license_text),
            'cell_type_label': clean_text(group_name),
            'text_blob': text_blob,
            'paper_context_blob': paper_context_blob,
            'text_with_context': ' '.join(part for part in [text_blob, paper_context_blob] if part),
            'evidence_sentences': sentences,
            'gene_names': gene_names,
            'gene_ids': gene_ids,
            'n_genes': len(gene_names),
            'n_sentences': len(sentences),
        }
    )

profiles = pd.DataFrame.from_records(records)
profiles = profiles[profiles['n_genes'] > 0].reset_index(drop=True)

print(f'Built {len(profiles):,} paper-celltype profiles.')
summary = pd.DataFrame(
    {
        'stat': ['papers', 'profiles', 'papers with abstract context', 'median genes/profile', 'median evidence sentences/profile'],
        'value': [
            profiles['paper_id'].nunique(),
            len(profiles),
            int((profiles['paper_context_blob'].str.len() > 0).sum()),
            int(profiles['n_genes'].median()),
            int(profiles['n_sentences'].median()),
        ],
    }
)
display(summary)
display(
    profiles[['cell_type_label', 'n_genes', 'n_sentences', 'title']]
    .sort_values(['n_genes', 'n_sentences'], ascending=False)
    .head(10)
)


Loaded title/abstract context for 504 papers from MECA XML.


Built 3,729 paper-celltype profiles.


,stat,value
0,papers,422
1,profiles,3729
2,papers with abstract context,3729
3,median genes/profile,2
4,median evidence sentences/profile,1


,cell_type_label,n_genes,n_sentences,title
1673,EPI,61,21,"Single-cell transcriptome analysis of human, m..."
1221,SMC8-MAC,51,8,Macrophage-like vascular smooth muscle cells d...
1757,AMD/RPD−,48,7,Patient induced pluripotent stem cells identif...
1675,PRE,42,15,"Single-cell transcriptome analysis of human, m..."
1920,NEUTROPHILS,41,25,Bronchoalveolar Lavage Single-Cell Transcripto...
2707,CD4+ NAÏVE,39,12,Immunological Differences in Atopic Dermatitis...
277,VESTIBULAR HC,38,10,The Dual Molecular Identity of Vestibular Kino...
2472,CD8+ T CELL,36,6,Antigen specificity of clonally-enriched CD8+ ...
1756,AMD/RPD+,35,10,Patient induced pluripotent stem cells identif...
276,TYPE II HC,35,7,The Dual Molecular Identity of Vestibular Kino...


## Retrieval setup

The current website uses a lightweight text retriever that can run entirely in the browser.

- The main profile text is `cell type label + evidence sentences`
- A smaller context field stores `title + abstract`
- Text retrieval uses TF-IDF cosine similarity as a sparse baseline
- Gene retrieval uses Jaccard similarity on the marker set

This section keeps that baseline and then compares it to a cached biomedical embedding model.

In [4]:
def normalize_profile_text(text: str) -> str:
    text = str(text).upper()
    text = text.replace('+', ' PLUS ').replace('-', ' MINUS ')
    text = re.sub(r'[^A-Z0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def token_set(text: str) -> set[str]:
    return set(re.findall(r'[A-Z0-9]+', normalize_profile_text(text)))


profiles['label_token_set'] = profiles['cell_type_label'].map(token_set)
profiles['text_token_set'] = profiles['text_blob'].map(token_set)

vectorizer = TfidfVectorizer(
    preprocessor=normalize_profile_text,
    token_pattern=r'(?u)\b[A-Z0-9]+\b',
    ngram_range=(1, 2),
    min_df=2,
)
text_matrix = vectorizer.fit_transform(profiles['text_blob'])
context_vectorizer = TfidfVectorizer(
    preprocessor=normalize_profile_text,
    token_pattern=r'(?u)\b[A-Z0-9]+\b',
    ngram_range=(1, 2),
    min_df=2,
)
context_matrix = context_vectorizer.fit_transform(profiles['paper_context_blob'])
print(f'Text vocabulary size: {len(vectorizer.vocabulary_):,}')
print(f'Context vocabulary size: {len(context_vectorizer.vocabulary_):,}')

DEFAULT_PROFILE_WEIGHT = 0.9
DEFAULT_CONTEXT_WEIGHT = 0.1
PROFILE_RESULT_MIN_SCORE = 0.2
print(f'Default hybrid weights: profile={DEFAULT_PROFILE_WEIGHT:.2f}, context={DEFAULT_CONTEXT_WEIGHT:.2f}')


def preview_sentence(sentences: list[str], max_len: int = 160) -> str:
    if not sentences:
        return ''
    text = sentences[0]
    return text if len(text) <= max_len else text[: max_len - 3] + '...'


def lexical_coverage(query_tokens: set[str], profile_tokens: set[str]) -> float:
    if not query_tokens:
        return 0.0
    return len(query_tokens & profile_tokens) / len(query_tokens)


def score_text_results(profile_scores: np.ndarray, context_scores: np.ndarray, query_text: str) -> np.ndarray:
    query_tokens = token_set(query_text)
    normalized_query = normalize_profile_text(query_text)
    label_coverage = profiles['label_token_set'].map(lambda s: lexical_coverage(query_tokens, s)).to_numpy()
    text_coverage = profiles['text_token_set'].map(lambda s: lexical_coverage(query_tokens, s)).to_numpy()
    label_hit = profiles['cell_type_label'].map(
        lambda value: 0.12 if normalize_profile_text(value).find(normalized_query) >= 0 else 0.0
    ).to_numpy()
    return (
        DEFAULT_PROFILE_WEIGHT * profile_scores
        + DEFAULT_CONTEXT_WEIGHT * context_scores
        + 0.22 * label_coverage
        + 0.05 * text_coverage
        + label_hit
    )


def format_text_result(scores: np.ndarray, profile_scores: np.ndarray, context_scores: np.ndarray, top_k: int = 8) -> pd.DataFrame:
    keep = np.flatnonzero(scores >= PROFILE_RESULT_MIN_SCORE)
    order = keep[np.argsort(scores[keep])[::-1][:top_k]] if len(keep) else np.array([], dtype=int)
    result = profiles.iloc[order][['cell_type_label', 'title', 'doi', 'n_genes', 'n_sentences']].copy()
    result.insert(0, 'score', scores[order])
    result['profile_score'] = profile_scores[order]
    result['context_score'] = context_scores[order]
    result['top_genes'] = [', '.join(profiles.iloc[i]['gene_names'][:6]) for i in order]
    result['evidence_preview'] = [preview_sentence(profiles.iloc[i]['evidence_sentences']) for i in order]
    return result.reset_index(drop=True)


def query_text(query_text: str, top_k: int = 8) -> pd.DataFrame:
    query_vec = vectorizer.transform([query_text])
    profile_scores = cosine_similarity(query_vec, text_matrix).ravel()
    context_query_vec = context_vectorizer.transform([query_text])
    context_scores = cosine_similarity(context_query_vec, context_matrix).ravel()
    scores = score_text_results(profile_scores, context_scores, query_text)
    return format_text_result(scores, profile_scores, context_scores, top_k=top_k)

Text vocabulary size: 38,007
Context vocabulary size: 66,244
Default hybrid weights: profile=0.90, context=0.10


In [5]:
text_queries = [
    't cells',
    'cytotoxic lymphocytes',
    'fibroblasts in stromal tissue',
]

for q in text_queries:
    print(f'\nTF-IDF TEXT QUERY: {q}')
    display(query_text(q, top_k=5))


TF-IDF TEXT QUERY: t cells


,score,cell_type_label,title,doi,n_genes,n_sentences,profile_score,context_score,top_genes,evidence_preview
0,0.815790,T CELLS,A single-cell immune atlas of primary and seco...,10.1101/2025.09.11.675613,5,1,0.472142,0.008620,"CD247, CD3D, CD3E, CD3G, ZAP70",T cells expressed pan-T cell marker CD3E and o...
1,0.801356,CYTOTOXIC ΑΒ T CELLS,A single-cell immune atlas of primary and seco...,10.1101/2025.09.11.675613,2,1,0.456105,0.008620,"GNLY, GZMB","In spleen, two populations of non-cycling αβ T..."
2,0.776189,NAÏVE T CELLS/STEM CELL MEMORY T CELLS,A Tonic Signaling Code Predicts CAR-T Cell Eff...,10.1101/2025.09.29.679095,2,1,0.420827,0.074448,"CD45RO, CD62L",Surface expression of CD62L and CD45RO was use...
3,0.741517,EFFECTOR T CELLS,A Tonic Signaling Code Predicts CAR-T Cell Eff...,10.1101/2025.09.29.679095,2,1,0.382303,0.074448,"CD45RO, CD62L",Surface expression of CD62L and CD45RO was use...
4,0.733078,ΓΔ T CELLS,A single-cell immune atlas of primary and seco...,10.1101/2025.09.11.675613,1,2,0.380240,0.008620,TRDC,"Variable expression of TRDC, encoding for a po..."



TF-IDF TEXT QUERY: cytotoxic lymphocytes


,score,cell_type_label,title,doi,n_genes,n_sentences,profile_score,context_score,top_genes,evidence_preview
0,0.410516,CYTOTOXIC PLASMA CELL,Revised International Staging System (R-ISS) s...,10.1101/2021.12.06.471423,5,1,0.299679,0.058048,"CCL4, CCL5, GNLY, GZMA, NKG7","The cytotoxic genes NKG7, GZMA, and GNLY and t..."
1,0.379153,CYTOTOXIC CD8+ T CELL,Multi-omics analysis reveals key immunogenic s...,10.1101/2024.11.28.625843,1,1,0.269802,0.013309,GZMB,Upregulation of the principal cytotoxic granzy...
2,0.369552,LYMPHOCYTES,MAVS mediates a protective immune response in ...,10.1101/2021.12.22.473954,6,1,0.260613,0.000000,"GZMB, IFI44, IFNG, IRF7, PRF1, XCL1",Upon examining genes that were differentially ...
3,0.357182,CYTOTOXIC,NK cells contribute to resistance to anti-PD1 ...,10.1101/2023.12.14.571631,3,1,0.246869,0.000000,"GNLY, GZMB, PRF1","In parallel, re-clustering identified four CD8..."
4,0.354207,PRE-CYTOTOXIC PHENOTYPE,Single cell profiling reveals functional heter...,10.1101/2022.01.24.477494,2,2,0.240097,0.031204,"CD107A, NKP46","Notably, no difference in CD107a expression up..."



TF-IDF TEXT QUERY: fibroblasts in stromal tissue


,score,cell_type_label,title,doi,n_genes,n_sentences,profile_score,context_score,top_genes,evidence_preview
0,0.235272,STROMAL CELL,Endocrine disruption of early uterine differen...,10.1101/2022.11.18.517135,4,2,0.170455,0.018620,"COL6A3, DPT, FOXL2, VCAN","Feature plots of three stromal cell markers, D..."
1,0.209587,TISSUE STEM CELL,Revealing the Organ-Specific Expression of SPT...,10.1101/2023.06.01.543198,2,2,0.125395,0.042316,"EPDR1, SPTBN1",EPDR1 is expressed in the Tissue stem cells of...


## Biomedical text embeddings

The repo already has a cached sentence-transformer model on disk:

- `pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb`

This section loads that model offline and uses it to embed the same two text fields stored for each profile:

- `text_blob = label repeated three times + title + evidence sentences`
- `paper_context_blob = title + abstract`

The goal is simple: test whether a biomedical text encoder gives better query behavior than the sparse browser baseline before changing the website.

In [6]:
os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')

CACHED_MODEL_PATHS = [
    Path.home() / '.cache' / 'torch' / 'sentence_transformers' / 'pritamdeka_BioBERT-mnli-snli-scinli-scitail-mednli-stsb',
    Path.home() / '.cache' / 'huggingface' / 'hub' / 'models--sentence-transformers--all-MiniLM-L6-v2',
]

for model_path in CACHED_MODEL_PATHS:
    if model_path.exists():
        BIOMED_MODEL_PATH = model_path
        break
else:
    raise FileNotFoundError('Could not find a cached sentence-transformer model on disk.')

print(f'Loading model from: {BIOMED_MODEL_PATH}')
biomed_model = SentenceTransformer(str(BIOMED_MODEL_PATH))
biomed_text_embeddings = biomed_model.encode(
    profiles['text_blob'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True,
)
biomed_context_embeddings = biomed_model.encode(
    profiles['paper_context_blob'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True,
)
print('Biomedical embedding shapes:', biomed_text_embeddings.shape, biomed_context_embeddings.shape)


def query_text_biomed(query_text: str, top_k: int = 8) -> pd.DataFrame:
    query_embedding = biomed_model.encode([query_text], normalize_embeddings=True)[0]
    profile_scores = biomed_text_embeddings @ query_embedding
    context_scores = biomed_context_embeddings @ query_embedding
    scores = score_text_results(profile_scores, context_scores, query_text)
    return format_text_result(scores, profile_scores, context_scores, top_k=top_k)

Loading model from: /Users/sinabooeshaghi/.cache/torch/sentence_transformers/pritamdeka_BioBERT-mnli-snli-scinli-scitail-mednli-stsb


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /Users/sinabooeshaghi/.cache/torch/sentence_transformers/pritamdeka_BioBERT-mnli-snli-scinli-scitail-mednli-stsb
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/117 [00:00<?, ?it/s]

Batches:   0%|          | 0/117 [00:00<?, ?it/s]

Biomedical embedding shapes: (3729, 768) (3729, 768)


## Smaller browser-friendly embedding model

A browser deployment likely needs a much smaller model than BioBERT. This notebook tests one practical candidate that is already cached locally:

- `sentence-transformers/all-MiniLM-L6-v2`

This model is general-domain rather than biomedical, but it is much smaller and closer to what a browser deployment could plausibly ship.

In [7]:
MINILM_MODEL_PATH = Path.home() / '.cache' / 'huggingface' / 'hub' / 'models--sentence-transformers--all-MiniLM-L6-v2' / 'snapshots' / 'c9745ed1d9f207416be6d2e6f8de32d1f16199bf'
assert MINILM_MODEL_PATH.exists(), f'Missing MiniLM snapshot: {MINILM_MODEL_PATH}'
print(f'Loading MiniLM from: {MINILM_MODEL_PATH}')
minilm_model = SentenceTransformer(str(MINILM_MODEL_PATH))
minilm_text_embeddings = minilm_model.encode(
    profiles['text_blob'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True,
)
minilm_context_embeddings = minilm_model.encode(
    profiles['paper_context_blob'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True,
)
print('MiniLM embedding shapes:', minilm_text_embeddings.shape, minilm_context_embeddings.shape)


def query_text_minilm(query_text: str, top_k: int = 8) -> pd.DataFrame:
    query_embedding = minilm_model.encode([query_text], normalize_embeddings=True)[0]
    profile_scores = minilm_text_embeddings @ query_embedding
    context_scores = minilm_context_embeddings @ query_embedding
    scores = score_text_results(profile_scores, context_scores, query_text)
    return format_text_result(scores, profile_scores, context_scores, top_k=top_k)

Loading MiniLM from: /Users/sinabooeshaghi/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /Users/sinabooeshaghi/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/117 [00:00<?, ?it/s]

Batches:   0%|          | 0/117 [00:00<?, ?it/s]

MiniLM embedding shapes: (3729, 384) (3729, 384)


## Browser baseline versus BioBERT versus MiniLM

We compare three text retrievers on the same query set.

- `tfidf_hybrid`: the current browser-safe baseline
- `biomed_hybrid`: cached BioBERT sentence-transformer
- `minilm_hybrid`: smaller cached MiniLM sentence-transformer

This is a qualitative check. The point is to see whether the smaller model is good enough to justify browser deployment.

In [8]:
comparison_queries = [
    'cytotoxic lymphocytes',
    'stromal fibroblasts',
    't cells',
    'tumor infiltrating t cells',
    'activated monocytes in blood',
]

for q in comparison_queries:
    print(f'\nCOMPARISON QUERY: {q}')
    tfidf_result = query_text(q, top_k=5).copy()
    tfidf_result.insert(0, 'retriever', 'tfidf_hybrid')
    biomed_result = query_text_biomed(q, top_k=5).copy()
    biomed_result.insert(0, 'retriever', 'biomed_hybrid')
    minilm_result = query_text_minilm(q, top_k=5).copy()
    minilm_result.insert(0, 'retriever', 'minilm_hybrid')
    display(pd.concat([tfidf_result, biomed_result, minilm_result], ignore_index=True))


COMPARISON QUERY: cytotoxic lymphocytes


,retriever,score,cell_type_label,title,doi,n_genes,n_sentences,profile_score,context_score,top_genes,evidence_preview
0,tfidf_hybrid,0.410516,CYTOTOXIC PLASMA CELL,Revised International Staging System (R-ISS) s...,10.1101/2021.12.06.471423,5,1,0.299679,0.058048,"CCL4, CCL5, GNLY, GZMA, NKG7","The cytotoxic genes NKG7, GZMA, and GNLY and t..."
1,tfidf_hybrid,0.379153,CYTOTOXIC CD8+ T CELL,Multi-omics analysis reveals key immunogenic s...,10.1101/2024.11.28.625843,1,1,0.269802,0.013309,GZMB,Upregulation of the principal cytotoxic granzy...
2,tfidf_hybrid,0.369552,LYMPHOCYTES,MAVS mediates a protective immune response in ...,10.1101/2021.12.22.473954,6,1,0.260613,0.000000,"GZMB, IFI44, IFNG, IRF7, PRF1, XCL1",Upon examining genes that were differentially ...
3,tfidf_hybrid,0.357182,CYTOTOXIC,NK cells contribute to resistance to anti-PD1 ...,10.1101/2023.12.14.571631,3,1,0.246869,0.000000,"GNLY, GZMB, PRF1","In parallel, re-clustering identified four CD8..."
4,tfidf_hybrid,0.354207,PRE-CYTOTOXIC PHENOTYPE,Single cell profiling reveals functional heter...,10.1101/2022.01.24.477494,2,2,0.240097,0.031204,"CD107A, NKP46","Notably, no difference in CD107a expression up..."
5,biomed_hybrid,0.773335,CYTOTOXIC T CELL,CD4 and CD8 co-receptors modulate functional a...,10.1101/2020.10.17.332072,2,1,0.658034,0.461044,"GRANZYMES, PERFORIN","Conversely, peptide-specific T cells that expr..."
6,biomed_hybrid,0.716053,MEMORY CYTOTOXIC T CELL,Human Immune Cell Epigenomic Signatures in Res...,10.1101/2023.06.29.546792,2,1,0.609139,0.328283,"CD3, CD8",The sorted cell types included Naive helper T ...
7,biomed_hybrid,0.710327,CYTOTOXIC CD8+ T CELL,Multi-omics analysis reveals key immunogenic s...,10.1101/2024.11.28.625843,1,1,0.606595,0.293913,GZMB,Upregulation of the principal cytotoxic granzy...
8,biomed_hybrid,0.702526,NAIVE CYTOTOXIC T CELL,Human Immune Cell Epigenomic Signatures in Res...,10.1101/2023.06.29.546792,4,1,0.594109,0.328283,"CCR7, CD3, CD45RA, CD8",The sorted cell types included Naive helper T ...
9,biomed_hybrid,0.696399,CYTOTOXIC T-CELL,Decontamination of ambient RNA in single-cell ...,10.1101/704015,1,1,0.605255,0.166701,CD8,Cell populations included progenitor cells(CD3...



COMPARISON QUERY: stromal fibroblasts


,retriever,score,cell_type_label,title,doi,n_genes,n_sentences,profile_score,context_score,top_genes,evidence_preview
0,tfidf_hybrid,0.476343,STROMAL CELL,Endocrine disruption of early uterine differen...,10.1101/2022.11.18.517135,4,2,0.374749,0.040684,"COL6A3, DPT, FOXL2, VCAN","Feature plots of three stromal cell markers, D..."
1,tfidf_hybrid,0.392650,STROMAL CELL,Spatial and Single-Cell Transcriptomics Deciph...,10.1101/2025.11.22.689892,1,1,0.286278,0.000000,MGP,"Expression patterns of PTPRC, MGP, and EPCAM d..."
2,tfidf_hybrid,0.337097,FIBROBLASTS,S3R: Modeling spatially varying associations w...,10.1101/2025.09.06.674629,1,1,0.222540,0.018105,VEGFA,"As an example, S3R identified VEGFA attributed..."
3,tfidf_hybrid,0.332568,MESENCHYMAL FIBROBLASTS,scRNA-seq reveals the diversity of the develop...,10.1101/2023.06.26.546508,4,2,0.219520,0.000000,"CDH1, CLDNH, COL1A1A, EPCAM",Two clusters constituting this cell lineage we...
4,tfidf_hybrid,0.331797,FIBROBLASTS,Patterning defects in mice with defective vent...,10.1101/2025.05.22.655570,1,1,0.218664,0.000000,PLK2,"Plk2 is also detected in fibroblasts, where it..."
5,biomed_hybrid,0.729684,MESENCHYMAL FIBROBLASTS,scRNA-seq reveals the diversity of the develop...,10.1101/2023.06.26.546508,4,2,0.632315,0.256005,"CDH1, CLDNH, COL1A1A, EPCAM",Two clusters constituting this cell lineage we...
6,biomed_hybrid,0.681447,ACTIVATED FIBROBLASTS,Critical role for the TGF-β1/mTORC1 signalling...,10.1101/2024.10.12.617979,2,1,0.549088,0.522670,"ACTA2, COL1A1",As well as the more traditional markers of act...
7,biomed_hybrid,0.658131,FIBROBLASTS,S3R: Modeling spatially varying associations w...,10.1101/2025.09.06.674629,1,1,0.543061,0.343770,VEGFA,"As an example, S3R identified VEGFA attributed..."
8,biomed_hybrid,0.645818,RETICULAR FIBROBLASTS,High-resolution integrative analysis allows ch...,10.1101/2025.07.29.667377,2,2,0.514183,0.480537,"CCL19, CD74","Of note, also KIT+ intestinal cells of Cajal (..."
9,biomed_hybrid,0.635990,RESIDENT FIBROBLASTS,scRNA-seq reveals the diversity of the develop...,10.1101/2023.06.26.546508,2,1,0.528210,0.256005,"CLDNH, EPCAM",Mesenchyme and fibroblasts make up a significa...



COMPARISON QUERY: t cells


,retriever,score,cell_type_label,title,doi,n_genes,n_sentences,profile_score,context_score,top_genes,evidence_preview
0,tfidf_hybrid,0.815790,T CELLS,A single-cell immune atlas of primary and seco...,10.1101/2025.09.11.675613,5,1,0.472142,0.008620,"CD247, CD3D, CD3E, CD3G, ZAP70",T cells expressed pan-T cell marker CD3E and o...
1,tfidf_hybrid,0.801356,CYTOTOXIC ΑΒ T CELLS,A single-cell immune atlas of primary and seco...,10.1101/2025.09.11.675613,2,1,0.456105,0.008620,"GNLY, GZMB","In spleen, two populations of non-cycling αβ T..."
2,tfidf_hybrid,0.776189,NAÏVE T CELLS/STEM CELL MEMORY T CELLS,A Tonic Signaling Code Predicts CAR-T Cell Eff...,10.1101/2025.09.29.679095,2,1,0.420827,0.074448,"CD45RO, CD62L",Surface expression of CD62L and CD45RO was use...
3,tfidf_hybrid,0.741517,EFFECTOR T CELLS,A Tonic Signaling Code Predicts CAR-T Cell Eff...,10.1101/2025.09.29.679095,2,1,0.382303,0.074448,"CD45RO, CD62L",Surface expression of CD62L and CD45RO was use...
4,tfidf_hybrid,0.733078,ΓΔ T CELLS,A single-cell immune atlas of primary and seco...,10.1101/2025.09.11.675613,1,2,0.380240,0.008620,TRDC,"Variable expression of TRDC, encoding for a po..."
5,biomed_hybrid,0.958385,T CELLS,Multiscale networks in multiple sclerosis,10.1101/2023.02.26.530153,1,1,0.613612,0.161338,CD3,The following cell populations were studied: T...
6,biomed_hybrid,0.917317,TOTAL T CELLS,Multiscale networks in multiple sclerosis,10.1101/2023.02.26.530153,3,2,0.567981,0.161338,"KS6B1, LCK, MK03",KS6B1 - LCK > Total T Cells - Th1 Non Classic ...
7,biomed_hybrid,0.916512,NAIVE CD8+ T CELLS,A single-cell transposable element atlas of hu...,10.1101/2023.12.28.573568,1,1,0.561601,0.210714,HARLEQUIN-17Q12,"Finally, CD4+ TCM, naive CD8+ T cells and CD8+..."
8,biomed_hybrid,0.916123,CENTRAL MEMORY T CELLS,A Tonic Signaling Code Predicts CAR-T Cell Eff...,10.1101/2025.09.29.679095,2,1,0.566582,0.161991,"CD45RO, CD62L",Surface expression of CD62L and CD45RO was use...
9,biomed_hybrid,0.908904,EFFECTOR MEMORY T CELLS,A Tonic Signaling Code Predicts CAR-T Cell Eff...,10.1101/2025.09.29.679095,2,1,0.558561,0.161991,"CD45RO, CD62L",Surface expression of CD62L and CD45RO was use...



COMPARISON QUERY: tumor infiltrating t cells


,retriever,score,cell_type_label,title,doi,n_genes,n_sentences,profile_score,context_score,top_genes,evidence_preview
0,tfidf_hybrid,0.382545,CEFX-SPECIFIC BRAIN METASTASIS-INFILTRATING PD...,The CD8+ T cell landscape of human brain metas...,10.1101/2021.08.03.455000,3,1,0.220197,0.243678,"IL7R, TCF7, TOX","Importantly, some of these experimentally-vali..."
1,tfidf_hybrid,0.369597,CD4+ T CELL,Dual inhibition of the nonsense mediated mRNA ...,10.64898/2025.12.04.692418,1,1,0.291303,0.024246,CD4,"Labels include: CD4+ T cells (CD4+T), regulato..."
2,tfidf_hybrid,0.344453,TUMOR,signifinder enables the identification of tumo...,10.1101/2023.03.07.530940,1,1,0.290956,0.025930,PD-L1,"In ductal breast carcinoma, the presence of a ..."
3,tfidf_hybrid,0.325009,T CELLS,A single-cell immune atlas of primary and seco...,10.1101/2025.09.11.675613,5,1,0.210628,0.004437,"CD247, CD3D, CD3E, CD3G, ZAP70",T cells expressed pan-T cell marker CD3E and o...
4,tfidf_hybrid,0.318570,CYTOTOXIC ΑΒ T CELLS,A single-cell immune atlas of primary and seco...,10.1101/2025.09.11.675613,2,1,0.203473,0.004437,"GNLY, GZMB","In spleen, two populations of non-cycling αβ T..."
5,biomed_hybrid,0.642712,T-CELL,Revealing the Organ-Specific Expression of SPT...,10.1101/2023.06.01.543198,1,1,0.605677,0.176021,SPTBN1,It is also highly expressed in the B-cell and ...
6,biomed_hybrid,0.616718,T CELLS,Multiscale networks in multiple sclerosis,10.1101/2023.02.26.530153,1,1,0.507189,0.252487,CD3,The following cell populations were studied: T...
7,biomed_hybrid,0.615091,CCL5- ΑΒ T CELLS,A single-cell immune atlas of primary and seco...,10.1101/2025.09.11.675613,2,1,0.505778,0.248909,"CCR7, S1PR1",Compared to CCL5+ counterparts (inferred to be...
8,biomed_hybrid,0.610757,T-REGULATORY CELL,Annotation-Free Prediction of Cancer Cells and...,10.1101/2025.11.09.687528,3,1,0.557563,0.289503,"CD3, CD4, FOXP3",By the cell type definition of T cells and mac...
9,biomed_hybrid,0.608875,T CELL,Single-Cell Transcriptomic Analysis of Kaposi ...,10.1101/2024.05.01.592010,3,1,0.536829,0.332288,"CD2, CD3E, CD8A",t-SNE plots of primary KS tumor samples highli...



COMPARISON QUERY: activated monocytes in blood


,retriever,score,cell_type_label,title,doi,n_genes,n_sentences,profile_score,context_score,top_genes,evidence_preview
0,tfidf_hybrid,0.244574,BLOOD NEUTROPHIL,Single-cell integration and multi-modal profil...,10.1101/2024.08.26.609563,1,1,0.178604,0.038310,CXCR2,"TAN markers (OLR1, PLAU, VEGFA) were highly up..."
1,tfidf_hybrid,0.209782,ACTIVATED MONOCYTE (HLA-DR+),Multiparameter stimulation mapping of signalin...,10.1101/2022.11.14.516371,1,1,0.129720,0.005338,HLA-DR,"In particular, the responding phenotype observ..."
2,tfidf_hybrid,0.206467,MONOCYTES,Identification of five characteristic ferropto...,10.1101/2023.07.20.549866,1,1,0.126153,0.004297,DUOX2,"In addition, spearman correlation analysis of ..."
3,biomed_hybrid,0.729575,ACTIVATED MONOCYTE (HLA-DR+),Multiparameter stimulation mapping of signalin...,10.1101/2022.11.14.516371,1,1,0.664189,0.393054,HLA-DR,"In particular, the responding phenotype observ..."
4,biomed_hybrid,0.689053,MONOCYTE,Single-cell integration and multi-modal profil...,10.1101/2024.08.26.609563,2,1,0.720790,0.278419,"CD14, VCAN",The myeloid lineage encompasses monocytes (VCA...
5,biomed_hybrid,0.681567,MONOCYTE-DERIVED MACROPHAGE,Donor monocyte-derived macrophages promote hum...,10.1101/787036,2,1,0.670426,0.656834,"S100A8/9, SIRPΑ","In GVHD, however, there was upregulation of mo..."
6,biomed_hybrid,0.678665,MONOCYTES,Depicting pathogenesis of osteomyelitis by sin...,10.1101/2025.01.23.634426,3,2,0.623018,0.379496,"CD11B, LY6C, PSTPIP2",Compared with WT and Morrbid−/−Pstpip2−/− mice...
7,biomed_hybrid,0.675548,MONOCYTES/MACROPHAGES,Bronchoalveolar Lavage Single-Cell Transcripto...,10.1101/2024.08.22.609137,2,1,0.638048,0.338043,"CD14, CD68","Utilizing key cellular markers, we identified ..."
8,minilm_hybrid,0.661120,ACTIVATED MONOCYTE (HLA-DR+),Multiparameter stimulation mapping of signalin...,10.1101/2022.11.14.516371,1,1,0.595276,0.328719,HLA-DR,"In particular, the responding phenotype observ..."
9,minilm_hybrid,0.645032,MONOCYTES,Single-cell transcriptome analysis of CD34+ st...,10.1101/438457,2,1,0.595713,0.288906,"CD14, CD68",In addition to separating cluster 6 into three...


### Selected default

The current website default still uses the lightweight browser-safe retriever.

- profile weight: `0.90`
- context weight: `0.10`
- minimum returned score: `0.20`

BioBERT gives stronger biological matches, but MiniLM is much smaller and closer to a realistic browser deployment target.

In [9]:
profiles['gene_name_set'] = profiles['gene_names'].map(set)
profiles['gene_id_set'] = profiles['gene_ids'].map(lambda ids: {g for g in ids if g})
profiles['gene_token_set'] = profiles.apply(lambda row: row['gene_name_set'] | row['gene_id_set'], axis=1)


def parse_gene_query(query: str) -> set[str]:
    tokens = re.split(r'[,;\s]+', query.upper().strip())
    return {token for token in tokens if token}


def jaccard(a: set[str], b: set[str]) -> float:
    if not a and not b:
        return 0.0
    union = a | b
    if not union:
        return 0.0
    return len(a & b) / len(union)


def query_genes(query: str, top_k: int = 8) -> pd.DataFrame:
    query_set = parse_gene_query(query)
    scores = profiles['gene_token_set'].map(lambda genes: jaccard(query_set, genes)).to_numpy()
    order = np.argsort(scores)[::-1][:top_k]
    result = profiles.iloc[order][['cell_type_label', 'title', 'doi', 'n_genes', 'n_sentences']].copy()
    result.insert(0, 'jaccard', scores[order])
    result['query_genes'] = ', '.join(sorted(query_set))
    result['top_genes'] = [', '.join(profiles.iloc[i]['gene_names'][:8]) for i in order]
    result['shared_matches'] = [', '.join(sorted(query_set & profiles.iloc[i]['gene_token_set'])) for i in order]
    result['evidence_preview'] = [preview_sentence(profiles.iloc[i]['evidence_sentences']) for i in order]
    return result.reset_index(drop=True)


In [10]:
gene_queries = [
    'IL7R LTB MALAT1',
    'ENSG00000168685',
    'ENSG00000105374 ENSG00000180644 ENSG00000115523',
    'DCN COL1A1 COL1A2',
]

for q in gene_queries:
    print(f'\nGENE QUERY: {q}')
    display(query_genes(q, top_k=5))


GENE QUERY: IL7R LTB MALAT1


,jaccard,cell_type_label,title,doi,n_genes,n_sentences,query_genes,top_genes,shared_matches,evidence_preview
0,0.333333,PRE-B,Stabilized marker gene identification and func...,10.1101/2024.08.21.608838,1,1,"IL7R, LTB, MALAT1",IL7R,IL7R,"Importantly, scSCOPE accurately identified Il7..."
1,0.333333,B-CELL,Stabilized marker gene identification and func...,10.1101/2024.08.21.608838,1,1,"IL7R, LTB, MALAT1",IL7R,IL7R,"As an example, scSCOPE identified Il7r, a gene..."
2,0.333333,LATE-PRO,Stabilized marker gene identification and func...,10.1101/2024.08.21.608838,1,1,"IL7R, LTB, MALAT1",IL7R,IL7R,"Importantly, scSCOPE accurately identified Il7..."
3,0.250000,IL7R+CD8_T,Tracing the stemness and malignant transition ...,10.1101/2025.04.06.647495,1,1,"IL7R, LTB, MALAT1",IL7R,IL7R,Six major T cell subtypes were identified: Naï...
4,0.250000,CD8-C3-IL7R,Single-cell atlas of tumor clonal evolution in...,10.1101/2020.08.18.254748,1,1,"IL7R, LTB, MALAT1",IL7R,IL7R,Large proportions of memory T cells of CD4-c2-...



GENE QUERY: ENSG00000168685


,jaccard,cell_type_label,title,doi,n_genes,n_sentences,query_genes,top_genes,shared_matches,evidence_preview
0,0.5,IL7RHIGHCD8+T,Tracing the stemness and malignant transition ...,10.1101/2025.04.06.647495,1,1,ENSG00000168685,IL7R,ENSG00000168685,Pseudotime trajectory analysis revealed T cell...
1,0.5,ILC1,Big data and single cell transcriptomics: impl...,10.1101/257352,1,1,ENSG00000168685,CD127,ENSG00000168685,Bjorklund et al. used scRNAseq to explore the ...
2,0.5,CD4 T CELL,Accurate highly variable gene selection using ...,10.1101/2025.06.23.661026,1,1,ENSG00000168685,IL7R,ENSG00000168685,"These include, for example, IL7R (CD4 T cells)..."
3,0.5,ILC3,Big data and single cell transcriptomics: impl...,10.1101/257352,1,1,ENSG00000168685,CD127,ENSG00000168685,Bjorklund et al. used scRNAseq to explore the ...
4,0.5,CD4+ T CELL,PBMC Fixation and Processing for Chromium Sing...,10.1101/315267,1,1,ENSG00000168685,IL7R,ENSG00000168685,Finer substructures were detected with the T-c...



GENE QUERY: ENSG00000105374 ENSG00000180644 ENSG00000115523


,jaccard,cell_type_label,title,doi,n_genes,n_sentences,query_genes,top_genes,shared_matches,evidence_preview
0,0.400000,NK-CELL GROUP,Decontamination of ambient RNA in single-cell ...,10.1101/704015,2,1,"ENSG00000105374, ENSG00000115523, ENSG00000180644","GNLY, NKG7","ENSG00000105374, ENSG00000115523",Cell clusters 2 and 3 were classified as B-cel...
1,0.400000,NK CELL,Comparative Analysis of Feature Selection Meth...,10.64898/2025.12.02.691907,2,1,"ENSG00000105374, ENSG00000115523, ENSG00000180644","GNLY, NKG7","ENSG00000105374, ENSG00000115523",Known marker recovery was evaluated for the PB...
2,0.300000,CD8.5,Acquisition of discrete immune suppressive bar...,10.1101/2024.12.31.630523,5,1,"ENSG00000105374, ENSG00000115523, ENSG00000180644","FCGR3A, FGFBP2, GNLY, NKG7, PRF1","ENSG00000105374, ENSG00000115523, ENSG00000180644",Clusters CD8.3 and CD8.5 with increased expres...
3,0.300000,CD8.3,Acquisition of discrete immune suppressive bar...,10.1101/2024.12.31.630523,5,1,"ENSG00000105374, ENSG00000115523, ENSG00000180644","FCGR3A, FGFBP2, GNLY, NKG7, PRF1","ENSG00000105374, ENSG00000115523, ENSG00000180644",Clusters CD8.3 and CD8.5 with increased expres...
4,0.285714,CD8+ T CELL,Single-cell RNA sequencing of peripheral blood...,10.1101/2024.01.04.574155,3,1,"ENSG00000105374, ENSG00000115523, ENSG00000180644","GNLY, GZMK, NKG7","ENSG00000105374, ENSG00000115523","In contrast, CD4+ cells expressed markedly hig..."



GENE QUERY: DCN COL1A1 COL1A2


,jaccard,cell_type_label,title,doi,n_genes,n_sentences,query_genes,top_genes,shared_matches,evidence_preview
0,0.5,MESENCHYMAL CELL,A comprehensive atlas of testicular interstiti...,10.1101/2024.08.02.606288,3,1,"COL1A1, COL1A2, DCN","COL1A2, COL3A1, DCN","COL1A2, DCN",Through uniform manifold approximation and pro...
1,0.5,FB,Single cell transcriptome analysis of cavernou...,10.1101/2023.05.25.542355,3,1,"COL1A1, COL1A2, DCN","COL1A1, COL1A2, FBLN1","COL1A1, COL1A2",The clusters were annotated by expression of m...
2,0.4,MKI67-MCS,A comprehensive atlas of testicular interstiti...,10.1101/2024.08.02.606288,4,1,"COL1A1, COL1A2, DCN","COL1A2, DCN, HELLS, LIG1","COL1A2, DCN","Additionally, we identified two proliferating ..."
3,0.4,FIBROBLAST,A multi-gene predictive model for the radiatio...,10.1101/2024.06.10.598247,2,1,"COL1A1, COL1A2, DCN","COL1A1, DCN","COL1A1, DCN",Fibroblasts expressed DCN and COL1A1 genes.
4,0.4,FIBROBLAST,Deciphering Tumor Microenvironment Dynamics in...,10.1101/2025.11.27.690370,2,1,"COL1A1, COL1A2, DCN","COL1A1, DCN","COL1A1, DCN",Annotation was supported by the expression of ...


In [11]:
biomed_dim = int(biomed_text_embeddings.shape[1])
n_profiles = len(profiles)
raw_float32_bytes = n_profiles * biomed_dim * 2 * 4
raw_float16_bytes = n_profiles * biomed_dim * 2 * 2
sample_n = min(100, n_profiles)
json_lengths = []
for i in range(sample_n):
    payload = json.dumps(np.round(biomed_text_embeddings[i], 6).tolist()) + json.dumps(np.round(biomed_context_embeddings[i], 6).tolist())
    json_lengths.append(len(payload))
json_estimate_bytes = int(np.mean(json_lengths) * n_profiles)
storage_df = pd.DataFrame(
    {
        'representation': ['float32 binary (text + context)', 'float16 binary (text + context)', 'rounded JSON estimate'],
        'approx_mb': [raw_float32_bytes / 1e6, raw_float16_bytes / 1e6, json_estimate_bytes / 1e6],
    }
)
display(storage_df)

,representation,approx_mb
0,float32 binary (text + context),22.910976
1,float16 binary (text + context),11.455488
2,rounded JSON estimate,126.704446


## Int8 profile vectors

The profile vectors are already unit-normalized. That makes a simple fixed-scale int8 test reasonable.

- quantize each embedding with `round(127 * x)`
- store as signed int8
- dequantize with `/ 127` for comparison

This is not model quantization. It only tests whether the stored profile vectors can be compressed aggressively without changing retrieval too much.

In [12]:
INT8_SCALE = 127.0

biomed_text_embeddings_int8 = np.clip(np.round(biomed_text_embeddings * INT8_SCALE), -127, 127).astype(np.int8)
biomed_context_embeddings_int8 = np.clip(np.round(biomed_context_embeddings * INT8_SCALE), -127, 127).astype(np.int8)

biomed_text_embeddings_int8_f = biomed_text_embeddings_int8.astype(np.float32) / INT8_SCALE
biomed_context_embeddings_int8_f = biomed_context_embeddings_int8.astype(np.float32) / INT8_SCALE


def query_text_biomed_int8(query_text: str, top_k: int = 8) -> pd.DataFrame:
    query_embedding = biomed_model.encode([query_text], normalize_embeddings=True)[0].astype(np.float32)
    profile_scores = biomed_text_embeddings_int8_f @ query_embedding
    context_scores = biomed_context_embeddings_int8_f @ query_embedding
    scores = score_text_results(profile_scores, context_scores, query_text)
    return format_text_result(scores, profile_scores, context_scores, top_k=top_k)

int8_queries = [
    'cytotoxic lymphocytes',
    'stromal fibroblasts',
    'activated monocytes in blood',
]

for q in int8_queries:
    print(f'\nINT8 COMPARISON QUERY: {q}')
    fp_result = query_text_biomed(q, top_k=5).copy()
    fp_result.insert(0, 'retriever', 'biomed_fp')
    int8_result = query_text_biomed_int8(q, top_k=5).copy()
    int8_result.insert(0, 'retriever', 'biomed_int8')
    display(pd.concat([fp_result, int8_result], ignore_index=True))


INT8 COMPARISON QUERY: cytotoxic lymphocytes


,retriever,score,cell_type_label,title,doi,n_genes,n_sentences,profile_score,context_score,top_genes,evidence_preview
0,biomed_fp,0.773335,CYTOTOXIC T CELL,CD4 and CD8 co-receptors modulate functional a...,10.1101/2020.10.17.332072,2,1,0.658034,0.461044,"GRANZYMES, PERFORIN","Conversely, peptide-specific T cells that expr..."
1,biomed_fp,0.716053,MEMORY CYTOTOXIC T CELL,Human Immune Cell Epigenomic Signatures in Res...,10.1101/2023.06.29.546792,2,1,0.609139,0.328283,"CD3, CD8",The sorted cell types included Naive helper T ...
2,biomed_fp,0.710327,CYTOTOXIC CD8+ T CELL,Multi-omics analysis reveals key immunogenic s...,10.1101/2024.11.28.625843,1,1,0.606595,0.293913,GZMB,Upregulation of the principal cytotoxic granzy...
3,biomed_fp,0.702526,NAIVE CYTOTOXIC T CELL,Human Immune Cell Epigenomic Signatures in Res...,10.1101/2023.06.29.546792,4,1,0.594109,0.328283,"CCR7, CD3, CD45RA, CD8",The sorted cell types included Naive helper T ...
4,biomed_fp,0.696399,CYTOTOXIC T-CELL,Decontamination of ambient RNA in single-cell ...,10.1101/704015,1,1,0.605255,0.166701,CD8,Cell populations included progenitor cells(CD3...
5,biomed_int8,0.775845,CYTOTOXIC T CELL,CD4 and CD8 co-receptors modulate functional a...,10.1101/2020.10.17.332072,2,1,0.660834,0.460943,"GRANZYMES, PERFORIN","Conversely, peptide-specific T cells that expr..."
6,biomed_int8,0.715127,MEMORY CYTOTOXIC T CELL,Human Immune Cell Epigenomic Signatures in Res...,10.1101/2023.06.29.546792,2,1,0.608056,0.328769,"CD3, CD8",The sorted cell types included Naive helper T ...
7,biomed_int8,0.711312,CYTOTOXIC CD8+ T CELL,Multi-omics analysis reveals key immunogenic s...,10.1101/2024.11.28.625843,1,1,0.607414,0.296401,GZMB,Upregulation of the principal cytotoxic granzy...
8,biomed_int8,0.701492,NAIVE CYTOTOXIC T CELL,Human Immune Cell Epigenomic Signatures in Res...,10.1101/2023.06.29.546792,4,1,0.592906,0.328769,"CCR7, CD3, CD45RA, CD8",The sorted cell types included Naive helper T ...
9,biomed_int8,0.701414,CYTOTOXIC T-CELL,Decontamination of ambient RNA in single-cell ...,10.1101/704015,1,1,0.610624,0.168528,CD8,Cell populations included progenitor cells(CD3...



INT8 COMPARISON QUERY: stromal fibroblasts


,retriever,score,cell_type_label,title,doi,n_genes,n_sentences,profile_score,context_score,top_genes,evidence_preview
0,biomed_fp,0.729684,MESENCHYMAL FIBROBLASTS,scRNA-seq reveals the diversity of the develop...,10.1101/2023.06.26.546508,4,2,0.632315,0.256005,"CDH1, CLDNH, COL1A1A, EPCAM",Two clusters constituting this cell lineage we...
1,biomed_fp,0.681447,ACTIVATED FIBROBLASTS,Critical role for the TGF-β1/mTORC1 signalling...,10.1101/2024.10.12.617979,2,1,0.549088,0.522670,"ACTA2, COL1A1",As well as the more traditional markers of act...
2,biomed_fp,0.658131,FIBROBLASTS,S3R: Modeling spatially varying associations w...,10.1101/2025.09.06.674629,1,1,0.543061,0.343770,VEGFA,"As an example, S3R identified VEGFA attributed..."
3,biomed_fp,0.645818,RETICULAR FIBROBLASTS,High-resolution integrative analysis allows ch...,10.1101/2025.07.29.667377,2,2,0.514183,0.480537,"CCL19, CD74","Of note, also KIT+ intestinal cells of Cajal (..."
4,biomed_fp,0.635990,RESIDENT FIBROBLASTS,scRNA-seq reveals the diversity of the develop...,10.1101/2023.06.26.546508,2,1,0.528210,0.256005,"CLDNH, EPCAM",Mesenchyme and fibroblasts make up a significa...
5,biomed_int8,0.728684,MESENCHYMAL FIBROBLASTS,scRNA-seq reveals the diversity of the develop...,10.1101/2023.06.26.546508,4,2,0.631342,0.254766,"CDH1, CLDNH, COL1A1A, EPCAM",Two clusters constituting this cell lineage we...
6,biomed_int8,0.680213,ACTIVATED FIBROBLASTS,Critical role for the TGF-β1/mTORC1 signalling...,10.1101/2024.10.12.617979,2,1,0.547843,0.521545,"ACTA2, COL1A1",As well as the more traditional markers of act...
7,biomed_int8,0.659989,FIBROBLASTS,S3R: Modeling spatially varying associations w...,10.1101/2025.09.06.674629,1,1,0.545109,0.343912,VEGFA,"As an example, S3R identified VEGFA attributed..."
8,biomed_int8,0.644236,RETICULAR FIBROBLASTS,High-resolution integrative analysis allows ch...,10.1101/2025.07.29.667377,2,2,0.512239,0.482211,"CCL19, CD74","Of note, also KIT+ intestinal cells of Cajal (..."
9,biomed_int8,0.632695,RESIDENT FIBROBLASTS,scRNA-seq reveals the diversity of the develop...,10.1101/2023.06.26.546508,2,1,0.524688,0.254766,"CLDNH, EPCAM",Mesenchyme and fibroblasts make up a significa...



INT8 COMPARISON QUERY: activated monocytes in blood


,retriever,score,cell_type_label,title,doi,n_genes,n_sentences,profile_score,context_score,top_genes,evidence_preview
0,biomed_fp,0.729575,ACTIVATED MONOCYTE (HLA-DR+),Multiparameter stimulation mapping of signalin...,10.1101/2022.11.14.516371,1,1,0.664189,0.393054,HLA-DR,"In particular, the responding phenotype observ..."
1,biomed_fp,0.689053,MONOCYTE,Single-cell integration and multi-modal profil...,10.1101/2024.08.26.609563,2,1,0.720790,0.278419,"CD14, VCAN",The myeloid lineage encompasses monocytes (VCA...
2,biomed_fp,0.681567,MONOCYTE-DERIVED MACROPHAGE,Donor monocyte-derived macrophages promote hum...,10.1101/787036,2,1,0.670426,0.656834,"S100A8/9, SIRPΑ","In GVHD, however, there was upregulation of mo..."
3,biomed_fp,0.678665,MONOCYTES,Depicting pathogenesis of osteomyelitis by sin...,10.1101/2025.01.23.634426,3,2,0.623018,0.379496,"CD11B, LY6C, PSTPIP2",Compared with WT and Morrbid−/−Pstpip2−/− mice...
4,biomed_fp,0.675548,MONOCYTES/MACROPHAGES,Bronchoalveolar Lavage Single-Cell Transcripto...,10.1101/2024.08.22.609137,2,1,0.638048,0.338043,"CD14, CD68","Utilizing key cellular markers, we identified ..."
5,biomed_int8,0.730106,ACTIVATED MONOCYTE (HLA-DR+),Multiparameter stimulation mapping of signalin...,10.1101/2022.11.14.516371,1,1,0.664707,0.393695,HLA-DR,"In particular, the responding phenotype observ..."
6,biomed_int8,0.688354,MONOCYTE,Single-cell integration and multi-modal profil...,10.1101/2024.08.26.609563,2,1,0.720223,0.276528,"CD14, VCAN",The myeloid lineage encompasses monocytes (VCA...
7,biomed_int8,0.679759,MONOCYTE-DERIVED MACROPHAGE,Donor monocyte-derived macrophages promote hum...,10.1101/787036,2,1,0.668195,0.658839,"S100A8/9, SIRPΑ","In GVHD, however, there was upregulation of mo..."
8,biomed_int8,0.677156,MONOCYTES,Depicting pathogenesis of osteomyelitis by sin...,10.1101/2025.01.23.634426,3,2,0.621450,0.378509,"CD11B, LY6C, PSTPIP2",Compared with WT and Morrbid−/−Pstpip2−/− mice...
9,biomed_int8,0.674942,MONOCYTES/MACROPHAGES,Bronchoalveolar Lavage Single-Cell Transcripto...,10.1101/2024.08.22.609137,2,1,0.637381,0.337988,"CD14, CD68","Utilizing key cellular markers, we identified ..."


## Export biomedical profile embeddings

This cell writes the biomedical profile embeddings into a dedicated SQLite table.

- table: `profile_embeddings_biomed`
- key: `(profile_id, model_name)`
- payloads:
  - `float16` binary blobs for `text_blob` and `paper_context_blob`
  - `int8` binary blobs for the same two fields

This keeps the export separate from the existing browser-safe embedding columns used by the current website.

In [13]:
MODEL_NAME_FP16 = 'pritamdeka_BioBERT-mnli-snli-scinli-scitail-mednli-stsb@float16'
MODEL_NAME_INT8 = 'pritamdeka_BioBERT-mnli-snli-scinli-scitail-mednli-stsb@int8'
MODEL_NAME_MINILM = 'sentence-transformers_all-MiniLM-L6-v2@float16'
EMBED_DTYPE_FP16 = np.dtype('<f2')
EMBED_DTYPE_INT8 = np.dtype('i1')

with sqlite3.connect(DB_PATH) as conn:
    sqlite_profiles = pd.read_sql_query(
        '''
        SELECT profile_id AS sqlite_profile_id, paper_id, group_name, collection
        FROM profiles
        WHERE collection = 'biorxiv'
        ''',
        conn,
    )

profile_export = profiles.merge(
    sqlite_profiles,
    left_on=['paper_id', 'cell_type_label'],
    right_on=['paper_id', 'group_name'],
    how='inner',
    validate='one_to_one',
)
assert len(profile_export) == len(profiles), 'Profile export join did not recover every bioRxiv profile.'

records_fp16 = [
    (
        int(row.sqlite_profile_id),
        MODEL_NAME_FP16,
        int(biomed_dim),
        'float16',
        np.asarray(text_vec, dtype=EMBED_DTYPE_FP16).tobytes(),
        np.asarray(context_vec, dtype=EMBED_DTYPE_FP16).tobytes(),
    )
    for row, text_vec, context_vec in zip(profile_export.itertuples(index=False), biomed_text_embeddings, biomed_context_embeddings)
]

records_int8 = [
    (
        int(row.sqlite_profile_id),
        MODEL_NAME_INT8,
        int(biomed_dim),
        'int8',
        np.asarray(text_vec, dtype=EMBED_DTYPE_INT8).tobytes(),
        np.asarray(context_vec, dtype=EMBED_DTYPE_INT8).tobytes(),
    )
    for row, text_vec, context_vec in zip(profile_export.itertuples(index=False), biomed_text_embeddings_int8, biomed_context_embeddings_int8)
]

minilm_dim = int(minilm_text_embeddings.shape[1])
records_minilm = [
    (
        int(row.sqlite_profile_id),
        MODEL_NAME_MINILM,
        int(minilm_dim),
        'float16',
        np.asarray(text_vec, dtype=EMBED_DTYPE_FP16).tobytes(),
        np.asarray(context_vec, dtype=EMBED_DTYPE_FP16).tobytes(),
    )
    for row, text_vec, context_vec in zip(profile_export.itertuples(index=False), minilm_text_embeddings, minilm_context_embeddings)
]

with sqlite3.connect(DB_PATH) as conn:
    conn.execute('PRAGMA foreign_keys = ON')
    conn.execute(
        '''
        CREATE TABLE IF NOT EXISTS profile_embeddings_biomed (
            profile_id INTEGER NOT NULL REFERENCES profiles(profile_id),
            model_name TEXT NOT NULL,
            dim INTEGER NOT NULL,
            dtype TEXT NOT NULL,
            text_embedding_blob BLOB NOT NULL,
            context_embedding_blob BLOB NOT NULL,
            PRIMARY KEY (profile_id, model_name)
        )
        '''
    )
    conn.execute(
        'DELETE FROM profile_embeddings_biomed WHERE model_name IN (?, ?, ?, ?)',
        (MODEL_NAME_FP16, MODEL_NAME_INT8, MODEL_NAME_MINILM, 'pritamdeka_BioBERT-mnli-snli-scinli-scitail-mednli-stsb')
    )
    conn.executemany(
        '''
        INSERT INTO profile_embeddings_biomed (
            profile_id,
            model_name,
            dim,
            dtype,
            text_embedding_blob,
            context_embedding_blob
        ) VALUES (?, ?, ?, ?, ?, ?)
        ''',
        records_fp16 + records_int8 + records_minilm,
    )
    conn.commit()
    exported = pd.read_sql_query(
        "SELECT model_name, dtype, COUNT(*) AS rows FROM profile_embeddings_biomed WHERE model_name IN (?, ?, ?) GROUP BY model_name, dtype ORDER BY model_name",
        conn,
        params=(MODEL_NAME_FP16, MODEL_NAME_INT8, MODEL_NAME_MINILM),
    )

display(exported)

,model_name,dtype,rows
0,pritamdeka_BioBERT-mnli-snli-scinli-scitail-me...,float16,3729
1,pritamdeka_BioBERT-mnli-snli-scinli-scitail-me...,int8,3729
2,sentence-transformers_all-MiniLM-L6-v2@float16,float16,3729


## What this MVP shows

This notebook now demonstrates six things.

- A profile-centric retrieval object works: text and gene queries both resolve to literature profiles.
- A cached biomedical sentence-transformer can be tested locally and compared directly to the browser-safe baseline.
- A much smaller MiniLM model can also be tested locally as a possible browser deployment target.
- Int8 profile vectors preserve much of the retrieval behavior while cutting vector storage in half relative to float16.
- BioBERT and MiniLM profile vectors can both be exported into SQLite in a dedicated table.
- The main deployment constraint is not database size. It is whether the website can run the same text encoder locally, without a server, for each user query.

In the current SQLite database there are 3,909 profiles. A 768-dimensional embedding for both `text_blob` and `paper_context_blob` requires about 22.9 MB as float32 binary, about 11.5 MB as float16 binary, and about 5.7 MB as int8 binary. The harder problem is query-time inference in the browser, not database storage.